# Notebook 01 :Nettoyage des données

**Projet :** AI Data Job Market — Prédiction de Salaire
**Fichier source :** `data/raw/AI_Job_Market_Dataset.csv`
**Fichier produit :** `data/processed/AI_Job_Market_Dataset_Cleaned.csv`

---
### Étapes de ce notebook
| # | Étape                                                        | Description |
|---|--------------------------------------------------------------|---|
| 1 | Importation                                                  | Charger les bibliothèques et le dataset |
| 2 | Nettoyage des noms de colonnes                               | minuscules, sans espaces  |
| 3 | Suppression colonnes inutiles  | `job_id`, `experience_level` |
| 4 | Nettoyage du texte                                           | strip, lowercase, espaces multiples |
| 5 | Suppression des doublons                                     | après le nettoyage texte |
| 6 | Variable cible `salary`                                      | NaN, valeurs ≤ 0, outliers IQR |
| 7 | `years_experience`                                           | NaN → médiane, type numérique |
| 8 | `hiring_urgency`                                             | encodage ordinal low=1 medium=2 high=3 |
| 9 | `company_size`                                               | mapping vers small/medium/large |
| 10 | `job_openings`                                               | vérification NaN et valeurs négatives |
| 11 | Feature Engineering                                          | création de `total_skills` |
| 12 | One-Hot Encoding                                             | variables catégorielles restantes |
| 13 | Vérification finale                                          | shape, NaN, types |
| 14 | Export                                                       | sauvegarde du CSV nettoyé |

## Étape 1 : Importation des bibliothèques et chargement des données

In [1]:
import pandas as pd
import numpy as np
import re
import os

#  Chargement
# os.path.join construit le chemin selon le système d'exploitation
# '..' remonte d'un dossier (de notebooks/ vers la racine du projet)
BASE_DIR = os.getcwd()

RAW_PATH  = os.path.join(BASE_DIR, '..', 'data', 'raw', 'AI_Job_Market_Dataset.csv')
OUT_PATH  = os.path.join(BASE_DIR, '..', 'data', 'processed', 'AI_Job_Market_Dataset_Cleaned.csv')

df = pd.read_csv(RAW_PATH)

print(f'Forme initiale : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')
print('\nAperçu :')
print(df.head(3))
print('\nTypes :')
print(df.dtypes)
print('\nValeurs manquantes par colonne :')
print(df.isnull().sum())

Forme initiale : (10345, 19)
Colonnes : ['job_id', 'job_title', 'company_size', 'company_industry', 'country', 'remote_type', 'experience_level', 'years_experience', 'education_level', 'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_month', 'job_posting_year', 'hiring_urgency', 'job_openings']

Aperçu :
   job_id                  job_title company_size company_industry    country  \
0       1                AI Engineer      Startup           Retail     Canada   
1       2  Machine Learning Engineer          MNC       Technology  Australia   
2       3  Machine Learning Engineer          MNC       Technology    Germany   

  remote_type experience_level  years_experience education_level  \
0      Remote           Senior                 2          Master   
1      Hybrid              Mid                 0        Bachelor   
2      Onsite              Mid                14          Master   

   skills_python  skills_sql  skills_m

## Étape 2 : Nettoyage des noms de colonnes

On transforme tous les noms de colonnes en **minuscules sans espaces**.
Cela évite les erreurs du type `df['Job Title']` vs `df['job_title']`.

In [2]:
df.columns = (
    df.columns
    .str.strip()         # supprime les espaces en début et fin
    .str.lower()         # tout en minuscules
    .str.replace(' ', '_', regex=False)  # espaces → underscores
)

print('Noms de colonnes après nettoyage :')
print(df.columns.tolist())


Noms de colonnes après nettoyage :
['job_id', 'job_title', 'company_size', 'company_industry', 'country', 'remote_type', 'experience_level', 'years_experience', 'education_level', 'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_month', 'job_posting_year', 'hiring_urgency', 'job_openings']


## Étape 3 : Suppression des colonnes inutiles

On supprime immédiatement **deux colonnes** :

- **`job_id`** : identifiant unique, aucune valeur analytique pour prédire un salaire.
- **`experience_level`** : catégorie texte (Entry/Mid/Senior). Dans ce dataset, cette variable
  n'est **pas cohérente** avec `years_experience` (les trois niveaux ont la même distribution
  d'années). On garde uniquement `years_experience` qui est numérique et directement utilisable
  par le modèle ML.


In [3]:
cols_to_drop = ['job_id', 'experience_level']

df.drop(columns=[c for c in cols_to_drop if c in df.columns],
        inplace=True)

print(f'Forme après suppression : {df.shape}')

Forme après suppression : (10345, 17)


## Étape 4 : Nettoyage des colonnes texte

Pour toutes les colonnes de type texte, on applique :
1. `.strip()` → supprime les espaces invisibles au début et à la fin
2. `.lower()` → tout en minuscules (évite que `'Data Scientist'` et `'data scientist'` soient traités comme deux valeurs différentes)
3. `re.sub(r'\s+', ' ', x)` → remplace plusieurs espaces consécutifs par un seul

**Cette étape doit être faite AVANT la suppression des doublons**,
sinon deux lignes identiques avec des casses différentes ne seraient pas détectées comme doublons.

In [4]:
# Sélection des colonnes textuelles
text_cols = df.select_dtypes(include=['object', 'string']).columns

# Nettoyage des colonnes textuelles
for col in text_cols:
    df[col] = (
        df[col]
        .astype('string')                     # garder les NaN correctement
        .str.strip()                          # supprimer espaces début/fin
        .str.lower()                          # minuscule
        .str.replace(r'\s+', ' ', regex=True) # espaces multiples → 1 seul
        .str.replace(r'[^\w\s]', '', regex=True) # supprimer ponctuation
    )


## Étape 5 : Suppression des doublons

On supprime les lignes **entièrement identiques**.  
Cette étape est placée **après** le nettoyage texte (étape 4) pour que des lignes  
comme `'Data Scientist'` et `'data scientist'` soient bien reconnues comme identiques.

In [5]:
avant = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {avant - len(df)}')
print(f'Forme après suppression : {df.shape}')

Doublons supprimés : 0
Forme après suppression : (10345, 17)


## Étape 6 : Nettoyage de la variable cible — `salary`

`salary` est la variable qu'on cherche à prédire. Elle doit être propre en priorité.

On effectue trois vérifications dans l'ordre :
1. **NaN** → on supprime les lignes sans salaire (on ne peut pas deviner le salaire)
2. **Valeurs ≤ 0** → un salaire nul ou négatif est une erreur de données
3. **Outliers (valeurs aberrantes)** → méthode IQR

### Rappel : mth. IQR
```
IQR  = Q3 - Q1
Borne basse  = Q1 - 1.5 × IQR
Borne haute  = Q3 + 1.5 × IQR
```
Toute valeur en dehors de cet intervalle est considérée comme aberrante et supprimée.

In [6]:
print('=== Analyse de salary ===')
print(df['salary'].describe())
print(f'NaN         : {df["salary"].isnull().sum()}')
print(f'Valeurs ≤ 0 : {(df["salary"] <= 0).sum()}')

print('=== Analyse de salary ===')

# Assurer type numérique
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')

print(df['salary'].describe())
print(f'NaN         : {df["salary"].isnull().sum()}')
print(f'Valeurs ≤ 0 : {(df["salary"] <= 0).sum()}')

# 1 Supprimer NaN
df = df.dropna(subset=['salary'])

# 2 Supprimer valeurs ≤ 0
df = df[df['salary'] > 0]

# 3 Supprimer outliers (IQR)
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

borne_basse = Q1 - 1.5 * IQR
borne_haute = Q3 + 1.5 * IQR

avant = len(df)

df = df[(df['salary'] >= borne_basse) & (df['salary'] <= borne_haute)]

print(f'\nBorne basse IQR : {borne_basse:.0f}')
print(f'Borne haute IQR : {borne_haute:.0f}')
print(f'Outliers supprimés : {avant - len(df)} lignes')
print(f'Forme après nettoyage salary : {df.shape}')

=== Analyse de salary ===
count     10345.00000
mean     113438.22726
std       31389.20106
min       45083.00000
25%       89715.00000
50%      113082.00000
75%      134894.00000
max      204143.00000
Name: salary, dtype: float64
NaN         : 0
Valeurs ≤ 0 : 0
=== Analyse de salary ===
count     10345.00000
mean     113438.22726
std       31389.20106
min       45083.00000
25%       89715.00000
50%      113082.00000
75%      134894.00000
max      204143.00000
Name: salary, dtype: float64
NaN         : 0
Valeurs ≤ 0 : 0

Borne basse IQR : 21946
Borne haute IQR : 202662
Outliers supprimés : 4 lignes
Forme après nettoyage salary : (10341, 17)


## Étape 7 : Nettoyage de `years_experience`

**Décision prise** : on garde `years_experience` telle quelle et on supprime `experience_level`  
(déjà fait à l'étape 2).

Actions :
1. Forcer le type numérique avec `pd.to_numeric()` — si une valeur est du texte par erreur, elle deviendra NaN
2. Vérifier le nombre de NaN → s'il y en a, on remplace par la **médiane** (plus robuste que la moyenne face aux valeurs extrêmes)
3. Vérifier les valeurs = 0 (valides pour les débutants)

**Pourquoi la médiane plutôt que la moyenne ?**  
La moyenne est sensible aux valeurs extrêmes. Si une personne a 50 ans d'expérience par erreur,  
la moyenne monte. La médiane reste stable.

In [7]:
# Forcer le type numérique
df['years_experience'] = pd.to_numeric(df['years_experience'], errors='coerce')

nan_count = df['years_experience'].isnull().sum()
print(f'NaN dans years_experience : {nan_count}')
print(f'Valeurs = 0              : {(df["years_experience"] == 0).sum()}')

# Remplacement des NaN par la médiane (si nécessaire)
if nan_count > 0:
    mediane = df['years_experience'].median()
    df['years_experience'] = df['years_experience'].fillna(mediane)
    print(f'{nan_count} NaN remplacés par la médiane : {mediane}')
else:
    print('Aucun NaN → aucune imputation nécessaire.')

print('\nDistribution finale de years_experience :')
print(df['years_experience'].describe())

NaN dans years_experience : 0
Valeurs = 0              : 707
Aucun NaN → aucune imputation nécessaire.

Distribution finale de years_experience :
count    10341.000000
mean         6.951552
std          4.320062
min          0.000000
25%          3.000000
50%          7.000000
75%         11.000000
max         14.000000
Name: years_experience, dtype: float64


## Étape 8 : Encodage ordinal de `hiring_urgency`

`hiring_urgency` a un **ordre logique** : Low < Medium < High.

On utilise un encodage **ordinal** (numéros qui respectent l'ordre) plutôt que le One-Hot Encoding  
(qui ignorerait l'ordre) :
```
low    → 1
medium → 2
high   → 3
```

 `.map()` transforme en NaN toute valeur absente du dictionnaire.  
On vérifie donc après le mapping et on remplace les éventuels NaN par la médiane.

In [8]:
df['hiring_urgency'] = df['hiring_urgency'].map({'low': 1, 'medium': 2, 'high': 3})

nan_urgency = df['hiring_urgency'].isnull().sum()
print(f'NaN après mapping : {nan_urgency}')

if nan_urgency > 0:
    df['hiring_urgency'] = df['hiring_urgency'].fillna(df['hiring_urgency'].median())
    print(f'{nan_urgency} NaN remplacés par la médiane.')

print('Valeurs hiring_urgency :', sorted(df['hiring_urgency'].unique()))

NaN après mapping : 0
Valeurs hiring_urgency : [np.int64(1), np.int64(2), np.int64(3)]


## Étape 9 : Standardisation de `company_size`

Le dataset mélange des **types** et des **tailles** d'entreprises dans la même colonne.  
On harmonise en 3 catégories de taille :

| Valeur originale | Signification | Nouvelle valeur |
|---|---|---|
| startup | Petite entreprise jeune | `small` |
| medium | Entreprise de taille intermédiaire | `medium` |
| mnc | Multinationale (Multi-National Corporation) | `large` |
| enterprise | Grande entreprise établie | `large` |

MNC et Enterprise deviennent toutes deux `large` car elles représentent des grandes structures.

In [9]:
mapping_company_size = {
    'startup'   : 'small',
    'medium'    : 'medium',
    'mnc'       : 'large',
    'enterprise': 'large'
}

df['company_size'] = df['company_size'].replace(mapping_company_size)

print('Valeurs company_size après mapping :')
print(df['company_size'].value_counts())

Valeurs company_size après mapping :
company_size
large     5137
small     2656
medium    2548
Name: count, dtype: Int64


## Étape 10 : Vérification de `job_openings` et `job_posting_year`

- **`job_openings`** : nombre de postes ouverts — doit être ≥ 0 et sans NaN.
- **`job_posting_year`** : on conserve toutes les années telles quelles (2020–2026),  
  y compris les années futures qui font partie de la distribution du dataset.

In [10]:
# job_openings
print('=== job_openings ===')
print(f'NaN        : {df["job_openings"].isnull().sum()}')
print(f'Valeurs < 0 : {(df["job_openings"] < 0).sum()}')
df['job_openings'] = df['job_openings'].fillna(df['job_openings'].median())

# job_posting_year — aucune modification
print('\n=== job_posting_year ===')
print(df['job_posting_year'].value_counts().sort_index())

=== job_openings ===
NaN        : 0
Valeurs < 0 : 0

=== job_posting_year ===
job_posting_year
2020    1468
2021    1479
2022    1494
2023    1454
2024    1501
2025    1480
2026    1465
Name: count, dtype: int64


## Étape 11 : Feature Engineering — création de `total_skills`

**Feature Engineering** = créer une nouvelle variable à partir des variables existantes.

Les 5 colonnes `skills_*` sont binaires (0 ou 1). On les additionne pour obtenir  
le **nombre total de compétences requises** pour chaque poste.

```
total_skills = skills_python + skills_sql + skills_ml + skills_deep_learning + skills_cloud
```

**Pourquoi c'est utile ?** Un poste qui demande 4 compétences techniques est probablement  
mieux payé qu'un poste qui en demande 1. Cette variable capture cette information globalement.

In [11]:
skill_cols = [col for col in df.columns if 'skills_' in col]
print(f'Colonnes compétences ({len(skill_cols)}) : {skill_cols}')

df['total_skills'] = df[skill_cols].sum(axis=1)

print('\nDistribution de total_skills :')
print(df['total_skills'].value_counts().sort_index())

Colonnes compétences (5) : ['skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud']

Distribution de total_skills :
total_skills
0     290
1    1662
2    3186
3    3200
4    1662
5     341
Name: count, dtype: int64


## Étape 12 : One-Hot Encoding des variables catégorielles

Les modèles ML ne comprennent que les chiffres. Les variables texte restantes  
doivent être transformées en colonnes binaires (0/1).

**Variables à encoder :** `job_title`, `company_size`, `company_industry`, `country`,  
`remote_type`, `education_level`

**Paramètre `drop_first=True`** : supprime la première catégorie de chaque variable  
pour éviter la **multicolinéarité** (problème mathématique où deux colonnes contiennent  
la même information, ce qui perturbe le modèle ML).

Catégories de référence supprimées automatiquement par `drop_first` :
| Variable | Catégorie de référence supprimée |
|---|---|
| `job_title` | ai engineer |
| `company_size` | large |
| `company_industry` | e-commerce |
| `country` | australia |
| `remote_type` | hybrid |
| `education_level` | bachelor |

In [12]:
# Colonnes texte restantes à encoder
text_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Colonnes à encoder :', text_cols)

# Remplir les NaN éventuels par le mode avant encodage
for col in text_cols:
    mode_val = df[col].mode()
    if len(mode_val) > 0:
        df[col] = df[col].fillna(mode_val[0])

# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True, dtype=int)

print('\nColonnes après One-Hot Encoding :')
print(df.columns.tolist())
print(f'Nombre total de colonnes : {len(df.columns)}')

Colonnes à encoder : []

Colonnes après One-Hot Encoding :
['years_experience', 'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_month', 'job_posting_year', 'hiring_urgency', 'job_openings', 'total_skills', 'job_title_business analyst', 'job_title_data analyst', 'job_title_data engineer', 'job_title_data scientist', 'job_title_machine learning engineer', 'company_size_medium', 'company_size_small', 'company_industry_education', 'company_industry_finance', 'company_industry_healthcare', 'company_industry_retail', 'company_industry_technology', 'country_canada', 'country_germany', 'country_india', 'country_singapore', 'country_uk', 'country_usa', 'remote_type_onsite', 'remote_type_remote', 'education_level_master', 'education_level_phd']
Nombre total de colonnes : 34


## Étape 13 : Vérification finale

Avant d'exporter, on vérifie que le DS est propre :
- aucune valeur manquante
- toutes les colonnes sont numériques
- `salary` est en dernière colonne (convention ML)
- la distribution de `salary` est cohérente

In [13]:

print('RÉSUMÉ FINAL DU NETTOYAGE')

print(f'Forme finale               : {df.shape}')
print(f'Valeurs manquantes totales : {df.isnull().sum().sum()}')
print(f'Toutes colonnes numériques : {df.select_dtypes(include=["object"]).empty}')

# Déplacer salary en dernière colonne (convention ML)
cols = [col for col in df.columns if col != 'salary'] + ['salary']
df = df[cols]
print(f'salary en dernière colonne : {df.columns[-1] == "salary"}')

print('\nListe complète des colonnes :')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

print('\nDistribution finale de salary :')
print(df['salary'].describe())

print('\nTypes de données :')
print(df.dtypes.value_counts())

print('\nAperçu des 5 premières lignes :')
print(df.head())

RÉSUMÉ FINAL DU NETTOYAGE
Forme finale               : (10341, 34)
Valeurs manquantes totales : 0
Toutes colonnes numériques : True
salary en dernière colonne : True

Liste complète des colonnes :
   1. years_experience
   2. skills_python
   3. skills_sql
   4. skills_ml
   5. skills_deep_learning
   6. skills_cloud
   7. job_posting_month
   8. job_posting_year
   9. hiring_urgency
  10. job_openings
  11. total_skills
  12. job_title_business analyst
  13. job_title_data analyst
  14. job_title_data engineer
  15. job_title_data scientist
  16. job_title_machine learning engineer
  17. company_size_medium
  18. company_size_small
  19. company_industry_education
  20. company_industry_finance
  21. company_industry_healthcare
  22. company_industry_retail
  23. company_industry_technology
  24. country_canada
  25. country_germany
  26. country_india
  27. country_singapore
  28. country_uk
  29. country_usa
  30. remote_type_onsite
  31. remote_type_remote
  32. education_level_mas

## Étape 14 : Export du dataset nettoyé

On sauvegarde le dataset dans `data/processed/`.  
Ce fichier sera lu par les notebooks suivants (visualisation et ML).

**Note importante sur la normalisation :**  
La normalisation (`StandardScaler`) n'est **pas faite ici**.  
Elle sera appliquée dans le notebook ML (03), après la séparation train/test,  
car normaliser avant le split introduirait une **fuite d'information** (data leakage).

In [14]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)

print(f'Fichier sauvegardé : {OUT_PATH}')
print(f'   {df.shape[0]} lignes × {df.shape[1]} colonnes')

Fichier sauvegardé : C:\Users\Admin\Desktop\ai-job-salary-predictor\notebooks\..\data\processed\AI_Job_Market_Dataset_Cleaned.csv
   10341 lignes × 34 colonnes
